# Conifer Wood Classification — ResNet-18 (species level)

Trains the reduced **ResNet-18** (stages with 16/32/64/128 filters) to
classify **18 conifer species** from 90×720 patches of grayscale
cross-sectional wood micrographs.

**Evaluation protocol (random-specimen split):** one whole specimen per
species is randomly held out for testing; training uses the remaining
specimens. This is repeated over 5 seeds (`0, 5, 10, 20, 56`) and results
are reported as mean ± std. Patches from the same specimen never appear in
both train and test.

> Set `DATA_PATH` below to the location of the HDF5 database
> (`database_20240110_gray_450X720_rb_treated.h5`) — see the README
> for the expected data layout.

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import shuffle

from common.cnn import build_resnet18
from common.utils import (
    SPECIES,
    SEED_LIST,
    load_database,
    key_labels,
    get_rs_split_indices,
    load_patches,
    compute_class_weight,
    make_label_map,
    plot_confusion_matrix,
    make_adamw,
    summarize_runs,
)

## 1. Configuration

In [ ]:
# Path to the HDF5 database (see README for the expected layout).
# Copy the database into ./data/ or point this to its location, e.g.:
#   "../../07a_conifer_wood_classification_ver.1/database_20240110_gray_450X720_rb_treated.h5"
DATA_PATH = "data/database_20240110_gray_450X720_rb_treated.h5"

# Training hyperparameters
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32
NUM_EPOCHS = 100
CROP_HEIGHT = 90   # patch height in pixels (width is the full 720)
LEVEL = "species"  # "species" (18 classes) or "family" (6 classes)

SAVE_DIR = "./models_resnet18"
os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Load database

In [ ]:
dataset, data_keys = load_database(DATA_PATH)
spec_n_list = key_labels(data_keys, level=LEVEL)

print(f"{len(data_keys)} specimens in the database")
for key in data_keys[:5]:
    print(" ", key, dataset[key].shape)

## 3. Training loop — 5 random-specimen splits

For each seed: build the split, crop patches, train with class weighting,
restore the best checkpoint (by validation accuracy) and evaluate on the
held-out specimens with a normalized confusion matrix.

In [ ]:
label_map = make_label_map(LEVEL)
class_names = list(label_map.keys())
reports = []

for seed in SEED_LIST:
    np.random.seed(seed)

    # --- random-specimen split: one held-out specimen per species ---
    train_index, test_index = get_rs_split_indices(spec_n_list, SPECIES, seed)

    # --- crop patches and build label arrays ---
    X_train, y_train_str = load_patches(
        dataset, data_keys, train_index, level=LEVEL, crop_height=CROP_HEIGHT
    )
    X_test, y_test_str = load_patches(
        dataset, data_keys, test_index, level=LEVEL, crop_height=CROP_HEIGHT
    )

    X_train = np.expand_dims(X_train, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)
    y_train = np.int32(list(map(label_map.get, y_train_str)))
    y_test = np.int32(list(map(label_map.get, y_test_str)))

    X_train, y_train = shuffle(X_train, y_train, random_state=0)
    X_test, y_test = shuffle(X_test, y_test, random_state=0)

    class_weight = compute_class_weight(y_train)
    num_classes = len(np.unique(y_train))

    # --- model ---
    tf.keras.backend.clear_session()
    model = build_resnet18(input_shape=X_train.shape[1:], num_classes=num_classes)
    model.compile(
        optimizer=make_adamw(LEARNING_RATE, WEIGHT_DECAY),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

    checkpoint_filepath = os.path.join(SAVE_DIR, f"resnet18_{LEVEL}_seed{seed}.h5")
    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        checkpoint_filepath,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )

    # --- train ---
    model.fit(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        epochs=NUM_EPOCHS,
        verbose=0,
        validation_split=0.2,
        callbacks=[checkpoint_callback],
        class_weight=class_weight,
    )

    # --- evaluate on the held-out specimens ---
    model.load_weights(checkpoint_filepath)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    plt.figure(figsize=(13, 11))
    plot_confusion_matrix(
        confusion_matrix(y_test, y_pred),
        class_names,
        normalize=True,
        title=f"ResNet-18 — seed {seed}",
    )
    plt.savefig(os.path.join(SAVE_DIR, f"resnet18_{LEVEL}_seed{seed}.png"),
                bbox_inches="tight", pad_inches=0.1, dpi=200)
    plt.show()

    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)
    print(f"seed {seed}: accuracy = {report['accuracy']:.4f}")

## 4. Summary

In [ ]:
summary = summarize_runs(reports)

print(f"Accuracy  : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print(f"Weighted F1: {summary['f1_mean']:.4f} ± {summary['f1_std']:.4f}")
print("Per-seed accuracies:", [round(a, 4) for a in summary["accuracies"]])